This notebook is designed to load neutron data and any non-neutron data (i.e. pressure, current, etc.) in an experimental folder, time bin it, and export it to CSV.

## Initialization

In [ ]:
# Importing needed code

import re
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from functools import reduce
from math import log, sqrt
from pathlib import Path
from typing import Callable, Literal

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from data_processing import helpers
from data_processing.dataframe_validation import (
    BinningDataframeColumn,
    DetectorDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn,
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData,
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_event_time,
    calculate_timetag_hours,
    calculate_timetag_hours_polars,
)
from data_processing.loading.window_loading import (
    get_neutron_window_paths,
    load_side_borders,
)
from data_processing.paths import (  # get_report_root,
    get_exp_root,
    get_reactor_data_root,
)
from data_processing.processing.calibration import (
    CalibrationType,
    Detector,
    recalibrate,
    recalibrate_polars
)
from data_processing.processing.neutron_classification import classify
from data_processing.processing.neutron_window_strategy.abstract_strategy import (
    AbstractNeutronStrategy,
)
from data_processing.processing.neutron_window_strategy.strategy_factory import (
    NeutronStrategyFactory,
)
from data_processing.processing.slice_fitting import (
    find_failed_slices,
    get_psd_energy_histogram,
    get_psd_energy_histogram_polars,
    scan_histogram_slices,
)
from data_processing.reporting.plotting import (
    add_fit_window_to_plot,
    add_supertitle_to_figure,
    plot_classification,
    plot_multiple_classification,
    plot_scatter,
)
from data_processing.types import (
    BimodalBounds,
    BimodalParams,
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    SliceFitStyle,
    WindowBorders,
    WindowType,
)
from matplotlib.colors import LightSource, LinearSegmentedColormap
from pint import Quantity

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df.loc[:, bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, "Time", time_bins)

    binned_df = (
        df.groupby("Time Bin", as_index=False)[data_col]
        .agg(["mean", "std"])
        .copy()
    )
    binned_df.columns = selected_cols
    binned_df.loc[:, "Bin midpoint"] = binned_df.index.to_series().apply(
        lambda x: x.mid
    )
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df.loc[:, BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[
    ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION
]
NasaBorderKey = Literal[
    ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC
]


def get_nasa_loading_settings(calib_key: CalibrationKey) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int,
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS
        if left_border_type == 1
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(calib_key: CalibrationKey) -> str:
    file_name_prefix = (
        f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    )
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey,
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float,
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float,
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int,
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int,
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS
                if existing_left_border_version_input == 1
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix
            )
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path
            )
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float,
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    filter_input = helpers.get_input_with_default(
        "Do you want to use smoothing filter on the bottom border? [y/n, or press Enter for no]",
        "n",
        str,
    )
    are_using_filter = filter_input.lower() == "y"
    if are_using_filter:
        filter_window = helpers.get_input_with_default(
            """\
Enter size of smoothing filter window
Press Enter for default (21)
""",
            21,
            int,
        )
        filter_order = helpers.get_input_with_default(
            """\
Enter value for smoothing filter polyorder
Press Enter for default (3)
""",
            3,
            int,
        )
    else:
        filter_window = 1
        filter_order = 1
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound,
        use_filter=are_using_filter,
        filter_window=filter_window,
        filter_order=filter_order,
    )
    return settings


def get_n_distro_generation_settings() -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float,
    )
    settings = NeutronDistributionGenerationSettings(sigma=sigma)
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings,
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )

    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData,
    factory_fn: Callable[[], AbstractNeutronStrategy],
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def plot_cmap(cmap: mpl.colors.Colormap, plot_alpha: bool = False):
    x = np.linspace(0, 1, 1000)
    cmap_out = cmap(x)
    print(cmap_out.shape)

    r, g, b, a = np.split(cmap_out, 4, axis=1)

    fig, ax = plt.subplots()
    ax.plot(x, r, c="red", ls="solid", label="Red")
    ax.plot(x, g, c="green", ls="dashed", label="Green")
    ax.plot(x, b, c="blue", ls="dotted", label="Blue")
    if plot_alpha:
        ax.plot(x, a, c="grey")

    ax.grid(visible=True)
    ax.legend()

In [ ]:
def compare_frames(
    new_lf: pl.LazyFrame, legacy_lf: pl.LazyFrame
) -> tuple[bool, str, None | pl.LazyFrame]:
    new_schema = new_lf.collect_schema()
    legacy_schema = legacy_lf.collect_schema()
    if len(new_schema) != len(legacy_schema):
        msg = (
            "Schemas have unequal lengths:",
            f"Legacy={len(legacy_schema)},",
            f"New={len(new_schema)}",
        )
        return False, msg, None
    schema_pairs = [
        elem for elem in zip(new_schema.items(), legacy_schema.items())
    ]
    schema_compare = [
        (lcol == dcol and ltype == dtype)
        for ((lcol, ltype), (dcol, dtype)) in schema_pairs
    ]
    schemas_equal = all(schema_compare)
    if not schemas_equal:
        print("Unequal schemas")
        print("Legacy:", legacy_schema)
        print("New:", new_schema)
        msg = (
            "Schemas do not match\n",
            f"Legacy:\n{legacy_schema}\n",
            f"New:\n{new_schema}",
        )
        return False, msg, None
    join_by = [
        DetectorDataframeColumn.TIMETAG.value,
        DetectorDataframeColumn.CHANNEL.value,
    ]
    compare_lf = new_lf.join(
        legacy_lf,
        on=join_by,
        how="full",
        suffix="_df",
    )
    # col_names = [col_name for ((col_name, _), _) in schema_pairs]
    # comparison_names = [f"compare_{col_name}" for col_name in col_names]
    comp_details = [
        (col_name, col_type, f"compare_{col_name}")
        for (col_name, col_type) in new_schema.items()
    ]
    comparisons = [
        (
            (pl.col(col_name).is_close(pl.col(f"{col_name}_df")))
            if "Float" in str(col_type)
            else (pl.col(col_name) == pl.col(f"{col_name}_df"))
        ).alias(f"compare_{col_name}")
        for col_name, col_type, comp_name in comp_details
    ]
    comp_cols = [pl.col(comp_col_name) for _, _, comp_col_name in comp_details]
    col_compare = compare_lf.with_columns(comparisons)
    frame_compare = col_compare.select(
        pl.fold(
            acc=pl.lit(True), function=lambda acc, x: acc & x, exprs=comp_cols
        )
        .all()
        .alias("compare_all")
    )

    non_match_predicates = [
        (pl.col(col_name) != pl.col(f"{col_name}_df"))
        for col_name, *_ in comp_details
    ]
    non_match_lf = compare_lf.with_columns(comparisons).filter(
        pl.Expr.or_(*non_match_predicates)
    )

    lf_is_same = frame_compare.collect().item()
    msg = (
        "Legacy dataframe and new LazyFrame are identical"
        if lf_is_same
        else "New LazyFrame differs from legacy dataframe"
    )

    return lf_is_same, msg, non_match_lf

In [ ]:
DEFAULT_LOAD_PATH = Path("polars_compare.csv")
DEFAULT_SAVE_PATH = Path("polars_compare_out.csv")


def load_comparison_lf(file_path: Path | None = None) -> pl.LazyFrame:
    csvpath = file_path if file_path is not None else DEFAULT_LOAD_PATH
    id_schema = {"id": pl.String, "timestamp": pl.Datetime}

    if not csvpath.is_file():
        lf = pl.LazyFrame({"id": [], "timestamp": []}, id_schema)
    else:
        lf = pl.scan_csv(csvpath, schema_overrides=id_schema, null_values="")

    schema = lf.collect_schema()
    match_col_names = [
        col_name for col_name in schema if ".is_match" in col_name
    ]
    message_col_names = [
        col_name for col_name in schema if ".message" in col_name
    ]
    match_struct_names = sorted(
        [col_name.split(".")[0] for col_name in match_col_names]
    )
    message_struct_names = sorted(
        [col_name.split(".")[0] for col_name in message_col_names]
    )
    if match_struct_names != message_struct_names:
        raise ValueError(
            "Data file columns include partial or invalid struct data"
        )

    match_cast_exprs = [
        pl.col(col_name).cast(pl.Boolean) for col_name in match_col_names
    ]
    lf = lf.with_columns(match_cast_exprs)

    struct_exprs = [
        pl.struct(
            is_match=f"{col_name}.is_match", message=f"{col_name}.message"
        ).alias(col_name)
        for col_name in match_struct_names
    ]
    lf = (
        lf.with_columns(struct_exprs)
        .drop(match_col_names)
        .drop(message_col_names)
    )

    return lf


def record_comparison(
    lf: pl.LazyFrame,
    exp_id: str,
    timestamp: datetime,
    step_name: str,
    is_match: bool,
    message: str,
) -> pl.LazyFrame:
    # id_schema = {"id": pl.String, "timestamp": pl.Datetime}
    result_struct_type = pl.Struct(
        {"is_match": pl.Boolean, "message": pl.String}
    )

    lf_schema = lf.collect_schema()
    # print(lf_schema)
    if step_name not in lf_schema:
        # print(f"new column {step_name}")
        lf = lf.with_columns(pl.lit(None, result_struct_type).alias(step_name))
        lf_schema = lf.collect_schema()
        # print(lf_schema)

    new_row = {
        "id": exp_id,
        "timestamp": timestamp,
        step_name: {"is_match": is_match, "message": message},
    }
    empty_cells = {k: None for k in lf_schema if k not in new_row}
    new_row = {**new_row, **empty_cells}
    # print(new_row)

    new_row_lf = pl.LazyFrame(new_row, lf_schema)
    lf = lf.update(new_row_lf, on=["id", "timestamp"], how="full")
    return lf


def save_comparison_lf(
    lf: pl.LazyFrame,
    save_path: Path | None = None,
    load_path: Path | None = None,
):
    # print(lf.collect())
    savepath = save_path if save_path is not None else DEFAULT_SAVE_PATH
    loadpath = load_path if load_path is not None else DEFAULT_LOAD_PATH
    schema = lf.collect_schema()
    # print(schema)
    unnest_cols = [
        col_name
        for col_name, dtype in schema.items()
        if isinstance(dtype, pl.Struct)
    ]
    # print(unnest_cols)
    lf = lf.unnest(unnest_cols, separator=".")
    # print(lf.collect())
    lf.sink_csv(savepath)

    replace_path = savepath.replace(loadpath)
    print(
        f"Save path {savepath} replaced load path {loadpath} at {replace_path}"
    )

## Experiment ID Input

In [ ]:
experiment_ids = helpers.input_experiment_ids()

In [ ]:
experiment_ids

In [ ]:
calib_input = helpers.get_input_with_default(
    "Do you want to use new calibration? [y/n, or press Enter for yes]",
    "y",
    str,
)

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration
    else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = (
    ExperimentDataKey.NEW_CALIBRATION
    if is_new_calibration
    else ExperimentDataKey.CAEN_CALIBRATION
)

In [ ]:
default_type_input = 1  # More accurate calibration curve, as of 2025-12-16
calib_type_input = helpers.get_input_with_default(
    """\
Which calibration curve do you want to use?
1: Lin-Log (default)
2: Linear
Press Enter for default
""",
    default_type_input,
    int,
)

calib_types: dict[int, CalibrationType] = {
    1: CalibrationType.LOG_CURVE,
    2: CalibrationType.LINEAR,
}
calib_type = calib_types.get(calib_type_input, calib_types[default_type_input])

In [ ]:
max_detectors = 2
detectors = {Detector.ZERO: "EJ-309.1", Detector.ONE: "EJ-309.2"}
done = False
while not done:
    num_detectors = helpers.get_input_with_default(
        """\
How many detectors are being used?
Press Enter for default (1)
""",
        1,
        int,
    )
    if 1 <= num_detectors <= max_detectors:
        done = True
    else:
        print(f"Please enter a valid number of detectors (max {max_detectors})")
if num_detectors == 1:
    detector_code = helpers.get_input_required(
        """\
Which detector was used?
1: Original detector (detector 1)
2: New detector (detector 2)
""",
        [Detector.ZERO, Detector.ONE],
        lambda x: Detector(int(x) - 1),
    )
    detector_setup = {0: detector_code}
else:
    channels = [0, 1]
    detector_setup = {}
    for detector, label in detectors.items():
        input_prompt = (
            f"What channel number is used for detector {label}?\n"
            f"Use one of the following options: {channels}"
        )
        channel = helpers.get_input_required(input_prompt, channels, int)
        # remove chosen channel from channels
        detector_setup[channel] = detector
        channels.remove(channel)
print("Detector setup:")
for channel, detector in detector_setup.items():
    print(f"    Channel {channel}: {detectors[detector]}")

In [ ]:
default_fit_input = (
    2  # changed to peak finder mode, approved by Fatima 2024-07-18
)
fit_input = helpers.get_input_with_default(
    """\
Which bimodal fit type do you want to use?
1: Bounds based
2: Peak finder based (default)
Press Enter for default
""",
    default_fit_input,
    int,
)

fit_styles: dict[int, SliceFitStyle] = {1: "bounds", 2: "peak_finder"}
fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])

In [ ]:
# kind of window (Nasa, N distribution)
# load or generate
# specific settings for each condition to make namedtuple
# - generator settings (i.e. sigma, etc.)
# - file path prefix for loading
done = False
strategy_factory = NeutronStrategyFactory()

while not done:
    window_input = helpers.get_input_with_default(
        """\
Which neutron classification window do you want to use?
1: NASA window (default)
2: Neutron distribution window
Press Enter for default
""",
        1,
        int,
    )
    load_window_input = helpers.get_input_with_default(
        """\
Do you want to load the borders from the standard border file?
[y/n, or press Enter for no]
""",
        "n",
        str,
    )
    done = True
    will_load = load_window_input == "y"

    try:
        if window_input == 1:
            if will_load:
                settings = get_nasa_loading_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "nasa", True, settings
                )
            else:
                settings = get_nasa_generation_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "nasa", False, settings
                )
                pass
        elif window_input == 2:
            if will_load:
                settings = get_n_distro_loading_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "n_distro", True, settings
                )
            else:
                settings = get_n_distro_generation_settings()
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "n_distro", False, settings
                )
        else:
            print("Invalid classification window type given, please try again")
            done = False
    except ValueError as err:
        print("Problem found:")
        print(err)
        print("Please try again")
        done = False

experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {} for exp_id in experiment_ids
}
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn
)

In [ ]:
bin_length = helpers.get_input_with_default(
    "Enter bin length (in seconds), or press Enter for default (300 s)",
    300,
    int,
)
bin_string = f"{bin_length}s"

In [ ]:
analysis_timestamp = datetime.now().strftime("%Y-%m%b-%d-%H-%M-%S")
overall_settings = {
    "calibration_type": repr(calib_key),
    "fitting_style": fit_style,
    "window_settings": repr(settings),
    "bin_length": bin_length,
}
print(repr(settings))

In [ ]:
analysis_timestamp

## Data Loading and Initial Processing

### Neutron Data Processing

In [ ]:
timestamp = datetime.now(timezone.utc)
cmp_lf = load_comparison_lf()

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    df, lf = load_parquet_psd(exp_id)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = df
    polars_data = {}
    polars_data[ExperimentDataKey.UNCLASSIFIED] = lf
    exp_data["polars"] = polars_data

    lazified_df = pl.LazyFrame(df).with_columns(
        pl.col("CHANNEL").cast(pl.UInt8)
    )
    match, msg, details_lf = compare_frames(lf, lazified_df)
    if match:
        print(f"{exp_id}: New Polars pipeline matches legacy pipeline")
    else:
        print(f"{exp_id}: New pipeline does not match legacy")
        print(msg)
        if details_lf is not None:
            print(details_lf.collect_schema())
            cols = [
                "CHANNEL",
                "ENERGYSHORT",
                "ENERGY",
                "TIMETAG",
                "tail / total",
            ]
            legacy_cols = [f"{col}_df" for col in cols]
            collected_dfs = pl.collect_all(
                [
                    lf.count(),
                    details_lf.count(),
                    details_lf.head(10).select(cols),
                    details_lf.head(10).select(legacy_cols),
                ]
            )
            lf_count, bad_count, lf_bad_rows, df_bad_rows = collected_dfs
            df_count = df.count()
            print("Legacy row counts:", df_count)
            print("New row counts:", lf_count)
            print("Non-matching row counts:", bad_count)
            print("First 10 non-matching rows:")
            print("Legacy:", df_bad_rows)
            print("New:", lf_bad_rows)
    cmp_lf = record_comparison(
        cmp_lf,
        exp_id,
        timestamp,
        "1-load_data",
        match,
        "OK" if match else msg,
    )

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    polars_data = exp_data["polars"]
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_lf = polars_data[ExperimentDataKey.UNCLASSIFIED]

    unclassified_df = calculate_timetag_hours(unclassified_df)
    unclassified_lf = calculate_timetag_hours_polars(unclassified_lf)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df
    polars_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_lf
    exp_data["polars"] = polars_data

    lazified_df = pl.LazyFrame(unclassified_df).with_columns(
        pl.col("CHANNEL").cast(pl.UInt8)
    )
    match, msg, details_lf = compare_frames(unclassified_lf, lazified_df)
    if match:
        print(f"{exp_id}: New Polars pipeline matches legacy pipeline")
    else:
        print(f"{exp_id}: New pipeline does not match legacy")
        print(msg)
        if details_lf is not None:
            print(details_lf.collect_schema())
            # cols = [
            #     "CHANNEL",
            #     "ENERGYSHORT",
            #     "ENERGY",
            #     "TIMETAG",
            #     "tail / total",
            # ]
            # legacy_cols = [f"{col}_df" for col in cols]
            # collected_dfs = pl.collect_all(
            #     [
            #         lf.count(),
            #         details_lf.count(),
            #         details_lf.head(10).select(cols),
            #         details_lf.head(10).select(legacy_cols),
            #     ]
            # )
            # lf_count, bad_count, lf_bad_rows, df_bad_rows = collected_dfs
            # df_count = df.count()
            # print("Legacy row counts:", df_count)
            # print("New row counts:", lf_count)
            # print("Non-matching row counts:", bad_count)
            # print("First 10 non-matching rows:")
            # print("Legacy:", df_bad_rows)
            # print("New:", lf_bad_rows)
    cmp_lf = record_comparison(
        cmp_lf,
        exp_id,
        timestamp,
        "2-timetag_hours",
        match,
        "OK" if match else msg,
    )

In [ ]:
# Check channels
for exp_id, exp_data in experiment_neutron_data.items():
    polars_data = exp_data["polars"]
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_lf = polars_data[ExperimentDataKey.UNCLASSIFIED]

    channel_ids = unclassified_df[
        DetectorDataframeColumn.CHANNEL.value
    ].unique()
    if len(channel_ids) != len(detector_setup):
        print(
            f"Incorrect number of channels in {exp_id}:",
            f"setup has {len(detector_setup)},",
            f"data has {len(channel_ids)}",
        )
        input("Press Enter to stop execution")
        helpers.stop()
    for channel_id in channel_ids:
        if channel_id not in detector_setup.keys():
            print(f"Channel {channel_id} not found in detector setup")
            input("Press Enter to stop execution")
            helpers.stop()

    channel_ids_polars = (
        unclassified_lf.select(pl.col(DetectorDataframeColumn.CHANNEL.value))
        .unique()
        .collect()
    )
    # print(channel_ids_polars)
    # for channel_id in channel_ids_polars[DetectorDataframeColumn.CHANNEL.value]:
    #     print(channel_id)
    match = len(channel_ids) == len(detector_setup)
    if match:
        for channel_id in channel_ids_polars[
            DetectorDataframeColumn.CHANNEL.value
        ]:
            if channel_id not in detector_setup.keys():
                match = False
                msg = f"Channel id {channel_id} not found in detector setup"
                break
    else:
        msg = (
            f"Incorrect number of channels in {exp_id}: "
            + f"setup has {len(detector_setup)}, "
            + f"data has {len(channel_ids)}"
        )

    cmp_lf = record_comparison(
        cmp_lf,
        exp_id,
        timestamp,
        "3-channel_check",
        match,
        "OK" if match else msg,
    )

In [ ]:
# Separate by channels
for exp_id, exp_data in experiment_neutron_data.items():
    polars_data = exp_data["polars"]
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_lf = polars_data[ExperimentDataKey.UNCLASSIFIED]

    channel_dfs = {
        channel: {
            ExperimentDataKey.UNCLASSIFIED: unclassified_df.loc[
                unclassified_df[DetectorDataframeColumn.CHANNEL.value]
                == channel
            ].copy()
        }
        for channel in detector_setup.keys()
    }

    channel_lfs = {
        channel: {
            ExperimentDataKey.UNCLASSIFIED: unclassified_lf.filter(
                pl.col(DetectorDataframeColumn.CHANNEL.value) == channel
            )
        }
        for channel in detector_setup.keys()
    }

    exp_data[ExperimentDataKey.BY_CHANNEL] = channel_dfs
    polars_data[ExperimentDataKey.BY_CHANNEL] = channel_lfs
    exp_data["polars"] = polars_data

    matches = True
    for channel in detector_setup.keys():
        channel_df = channel_dfs[channel][ExperimentDataKey.UNCLASSIFIED]
        channel_lf = channel_lfs[channel][ExperimentDataKey.UNCLASSIFIED]
        lazified_df = pl.LazyFrame(channel_df).with_columns(
            pl.col("CHANNEL").cast(pl.UInt8)
        )
        match, msg, details_lf = compare_frames(channel_lf, lazified_df)
        if not match:
            matches = False
            print(
                f"{exp_id}: New pipeline does not match legacy on channel {channel}"
            )
            print(msg)
            if details_lf is not None:
                print(details_lf.collect_schema())
            break
    if matches:
        print(f"{exp_id}: New Polars pipeline matches legacy pipeline")

    cmp_lf = record_comparison(
        cmp_lf,
        exp_id,
        timestamp,
        "4-separate_channels",
        matches,
        "OK" if matches else msg,
    )

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    polars_data = exp_data["polars"]
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    channels_data_polars = polars_data[ExperimentDataKey.BY_CHANNEL]

    matches = True
    for channel in detector_setup.keys():
        print(exp_id, channel)
        unclassified_df = channels_data[channel][ExperimentDataKey.UNCLASSIFIED]
        unclassified_lf = channels_data_polars[channel][ExperimentDataKey.UNCLASSIFIED]
        
        recalibrated_df = recalibrate(
            unclassified_df, detector_setup[channel], calib_type
        )
        recalibrated_lf = recalibrate_polars(
            unclassified_lf, detector_setup[channel], calib_type
        )
        
        channels_data[channel][ExperimentDataKey.UNCLASSIFIED] = recalibrated_df
        channels_data_polars[channel][ExperimentDataKey.UNCLASSIFIED] = recalibrated_lf

        lazified_df = pl.LazyFrame(recalibrated_df).with_columns(
            pl.col("CHANNEL").cast(pl.UInt8)
        )
        match, msg, details_lf = compare_frames(recalibrated_lf, lazified_df)
        if not match:
            matches = False
            print(
                f"{exp_id}: New pipeline does not match legacy on channel {channel}"
            )
            print(f"{channel}: {msg}")
            if details_lf is not None:
                print(details_lf.collect_schema())
            break

    exp_data["polars"] = polars_data
    
    if matches:
        print(f"{exp_id}: New Polars pipeline matches legacy pipeline")

    cmp_lf = record_comparison(
        cmp_lf,
        exp_id,
        timestamp,
        "5-recalibration",
        matches,
        "OK" if matches else msg,
    )

In [ ]:
# Generate histogram

start_scan_idx = 0
# end_scan_idx = 420
end_scan_idx = 1000
energy_width = 5e-3
overall_settings["scan_idx"] = f"({start_scan_idx}, {end_scan_idx})"
overall_settings["energy_width"] = energy_width

for exp_id, exp_data in experiment_neutron_data.items():
    polars_data = exp_data["polars"]
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    channels_data_polars = polars_data[ExperimentDataKey.BY_CHANNEL]
    
    matches = True
    msg = ""
    shape_msg = ""
    for channel in detector_setup.keys():
        print(exp_id, channel)
        channel_data = channels_data[channel]
        channel_data_polars = channels_data_polars[channel]
        psd_report = channel_data[ExperimentDataKey.UNCLASSIFIED]
        psd_report_lf = channel_data_polars[ExperimentDataKey.UNCLASSIFIED]
        
        Z, xe, ye = get_psd_energy_histogram(
            psd_report, calibrated_energy_column, energy_width=energy_width
        )
        Z_p, xe_p, ye_p = get_psd_energy_histogram_polars(
            psd_report_lf, calibrated_energy_column, energy_width=energy_width
        )
        
        channel_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
        channel_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
        channel_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
        channel_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

        channel_data_polars[ExperimentDataKey.PSD_HISTOGRAM] = Z_p
        channel_data_polars[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe_p
        channel_data_polars[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye_p
        channel_data_polars[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z_p))

        channels_data[channel] = channel_data
        channels_data_polars[channel] = channel_data_polars

        Z_match = np.allclose(Z, Z_p, equal_nan=True)
        if not Z_match:
            matches = False
            msg = f"{channel}: Histograms do not match"
            shape_msg = f"Shapes: Legacy = {Z.shape}, New = {Z_p.shape}"
            break
        xe_match = np.allclose(xe, xe_p, equal_nan=True)
        if not xe_match:
            matches = False
            msg = f"{channel}: X edges do not match"
            shape_msg = f"Shapes: Legacy = {xe.shape}, New = {xe_p.shape}"
            break
        ye_match = np.allclose(ye, ye_p, equal_nan=True)
        if not ye_match:
            matches = False
            msg = f"{channel}: Y edges do not match"
            shape_msg = f"Shapes: Legacy = {ye.shape}, New = {ye_p.shape}"
            break

    exp_data[ExperimentDataKey.BY_CHANNEL] = channels_data
    polars_data[ExperimentDataKey.BY_CHANNEL] = channels_data_polars
    exp_data["polars"] = polars_data

    if matches:
        print(f"{exp_id}: New Polars pipeline matches legacy pipeline")
    else:
        print(
            f"{exp_id}: New pipeline does not match legacy on channel {channel}"
        )
        print(msg)
        print(shape_msg)

    cmp_lf = record_comparison(
        cmp_lf,
        exp_id,
        timestamp,
        "6-histogram",
        matches,
        "OK" if matches else msg,
    )

In [ ]:
save_comparison_lf(cmp_lf)

In [ ]:
# get fit dataframe
# (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        Z = channel_data[ExperimentDataKey.PSD_HISTOGRAM]
        xe = channel_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
        ye = channel_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
        end_scan_idx = channel_data[ExperimentDataKey.END_SCAN_IDX]

        if fit_style == "bounds":
            # Default
            default_bounds: BimodalBounds = (
                BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
                BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000),
            )

            bounds_a: BimodalBounds = (
                BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
                BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000),
            )

            bounds_b: BimodalBounds = (
                BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
                BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000),
            )

            # Ranged Example
            bounds = [
                ((0, 60), bounds_a),
            ]
        else:
            default_bounds = None
            bounds = None

        df, df_err = scan_histogram_slices(
            Z,
            xe,
            ye,
            fit_style=fit_style,
            default_bounds=default_bounds,
            bounds=bounds,
            start_idx=start_scan_idx,
            end_idx=end_scan_idx,
        )
        df, bad_slice_indexes = find_failed_slices(
            df, exp_id, nan_total_threshold=10
        )

        if bad_slice_indexes is not None:
            channel_data[ExperimentDataKey.VALID_SLICE_FITS] = df
            channel_data[ExperimentDataKey.BAD_SLICE_INDEXES] = (
                bad_slice_indexes
            )
            stop_here = True
        else:
            # channel_data['fom_results'] = df
            channel_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    helpers.stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]
    for channel, channel_data in channels_data.items():
        if ExperimentDataKey.FOM_RESULTS not in channel_data:
            print(f"No good fit data on Experiment {exp_id}")
            continue

        fom_results = channel_data[ExperimentDataKey.FOM_RESULTS]

        strategy.set_slice_fit_dataframe(fom_results)
        borders = strategy.get_neutron_window()

        channel_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        psd_report = channel_data[ExperimentDataKey.UNCLASSIFIED].copy()
        borders = channel_data[ExperimentDataKey.BORDERS]

        psd_report = classify(
            psd_report,
            calibrated_energy_column,
            borders,
            DetectorDataframeColumn.NEW_N_CLASS,
        )

        channel_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Get experiment start time
for exp_id, exp_data in experiment_neutron_data.items():
    exp_root = get_exp_root(exp_id)
    with open(exp_root / "exp_info.toml") as exp_info:
        exp_start_line = [line for line in exp_info if "exp_start" in line][0]
    exp_start_text = exp_start_line.replace("exp_start = ", "").strip()
    exp_start = datetime.fromisoformat(exp_start_text).astimezone(timezone.utc)
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        channel_data[ExperimentDataKey.START_TIME] = exp_start

In [ ]:
# Get timetag as clock time
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        psd_report = channel_data[ExperimentDataKey.PSD_REPORT]
        exp_start = channel_data[ExperimentDataKey.START_TIME]

        psd_report = calculate_event_time(psd_report, exp_start)

        channel_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        neutron_report = channel_data[ExperimentDataKey.PSD_REPORT]
        max_timetag_hrs = neutron_report[
            DetectorDataframeColumn.TIME_HOURS.value
        ].max()
        print(exp_id, channel, max_timetag_hrs)

In [ ]:
# Separate neutron and gamma events
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        psd_report = channel_data[ExperimentDataKey.PSD_REPORT]

        n_classify_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
        neutrons_only = psd_report.query(n_classify_col_name).copy()
        gamma_only = psd_report.query(f"~{n_classify_col_name}").copy()
        channel_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
        channel_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

### Non-Neutron Data Processing

In [ ]:
# process reactor data files
# stored in reactor_data
# File name format: Device Param 00x
# If same device/param, but different numbers, should be merged

data_file_pattern = re.compile(r"([a-zA-Z ]+) (\d+)")
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        reactor_data_folder = get_reactor_data_root(exp_id)
        time_col_name = NonReactorDataframeColumn.TIME.value
        data_col_name = NonReactorDataframeColumn.DATA.value
        units_col_name = NonReactorDataframeColumn.UNITS.value

        if not reactor_data_folder.is_dir():
            continue

        reactor_data_files = defaultdict(list)
        for file in reactor_data_folder.iterdir():
            match = data_file_pattern.match(file.name)
            if match and len(match.groups()) > 0:
                data_source = match.group(1)
                if isinstance(data_source, str):
                    reactor_data_files[data_source].append(file)

        reactor_data = {}
        for data_source, files in reactor_data_files.items():
            file_dfs = [
                pd.read_csv(
                    file,
                    header=0,
                    dtype=str,
                    encoding="cp1252",
                    names=[time_col_name, data_col_name, units_col_name],
                )
                for file in sorted(files)
            ]
            for df in file_dfs:
                df[time_col_name] = pd.to_datetime(df[time_col_name], utc=True)
            df = pd.concat(file_dfs, ignore_index=True)
            reactor_data[data_source] = df

        channel_data[ExperimentDataKey.REACTOR_DATA] = reactor_data

## Data Binning

In [ ]:
# Create bins
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        # neutron_report = channel_data[ExperimentDataKey.NEUTRONS_ONLY]
        psd_report = channel_data[ExperimentDataKey.PSD_REPORT]

        event_time_col = DetectorDataframeColumn.EVENT_TIME.value
        # start_time = neutron_report[event_time_col].min()
        start_time = channel_data[ExperimentDataKey.START_TIME]
        end_time = psd_report[event_time_col].max()
        timetag_clock_bins = pd.date_range(
            start=start_time, end=end_time, freq=bin_string
        )
        if len(timetag_clock_bins) == 1:
            timetag_clock_bins = pd.date_range(
                start=start_time, end=end_time, periods=2
            )
            print(
                f"{exp_id}_{channel}:",
                "Dwell time longer than experimental time,",
                "making one dwell bin",
            )
        channel_data[ExperimentDataKey.TIME_BIN_EDGES] = timetag_clock_bins

In [ ]:
# Bin neutron data
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        neutron_report = channel_data[ExperimentDataKey.NEUTRONS_ONLY]
        time_bins = channel_data[ExperimentDataKey.TIME_BIN_EDGES]

        time_col_name = DetectorDataframeColumn.EVENT_TIME.value
        time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
        count_col_name = BinningDataframeColumn.COUNT.value
        count_error_col_name = BinningDataframeColumn.COUNT_ERROR.value
        bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
        n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
        n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

        start_time = time_bins[0]

        binned_neutrons = get_time_cut(neutron_report, time_col_name, time_bins)
        binned_neutrons = (
            neutron_report.groupby(
                time_bin_col_name, as_index=True, observed=False
            )
            .size()
            .to_frame()
            .copy()
        )
        binned_neutrons.columns = [count_col_name]
        binned_neutrons[count_error_col_name] = np.sqrt(
            binned_neutrons[count_col_name]
        )

        binned_neutron_time_bins = binned_neutrons.index.to_series()
        midpoints = binned_neutron_time_bins.apply(lambda x: x.mid)
        durations = binned_neutron_time_bins.apply(
            lambda x: x.length.total_seconds()
        ).astype(np.float64)

        binned_neutrons[bin_mid_col_name] = midpoints
        binned_neutrons = bin_midpoint_time_to_seconds(
            binned_neutrons, start_time
        )

        binned_neutrons[n_rate_col_name] = (
            binned_neutrons[count_col_name] / durations
        )
        binned_neutrons[n_error_col_name] = (
            binned_neutrons[count_error_col_name] / durations
        )
        binned_neutrons = binned_neutrons.drop(
            [count_col_name, count_error_col_name], axis=1
        ).copy()
        channel_data[ExperimentDataKey.BINNED_NEUTRONS] = binned_neutrons

In [ ]:
# Bin gamma data
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        gamma_report = channel_data[ExperimentDataKey.GAMMA_ONLY]
        time_bins = channel_data[ExperimentDataKey.TIME_BIN_EDGES]

        time_col_name = DetectorDataframeColumn.EVENT_TIME.value
        time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
        count_col_name = BinningDataframeColumn.COUNT.value
        count_error_col_name = BinningDataframeColumn.COUNT_ERROR.value
        bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
        g_rate_col_name = BinningDataframeColumn.GAMMA_RATE.value
        g_error_col_name = BinningDataframeColumn.GAMMA_RATE_ERROR.value

        start_time = time_bins[0]

        binned_gamma = get_time_cut(gamma_report, time_col_name, time_bins)
        binned_gamma = (
            gamma_report.groupby(
                time_bin_col_name, as_index=True, observed=False
            )
            .size()
            .to_frame()
            .copy()
        )
        binned_gamma.columns = [count_col_name]
        binned_gamma[count_error_col_name] = np.sqrt(
            binned_gamma[count_col_name]
        )

        binned_gamma_time_bins = binned_gamma.index.to_series()
        midpoints = binned_gamma_time_bins.apply(lambda x: x.mid)
        durations = binned_gamma_time_bins.apply(
            lambda x: x.length.total_seconds()
        ).astype(np.float64)

        binned_gamma[bin_mid_col_name] = midpoints
        binned_gamma = bin_midpoint_time_to_seconds(binned_gamma, start_time)

        binned_gamma[g_rate_col_name] = binned_gamma[count_col_name] / durations
        binned_gamma[g_error_col_name] = (
            binned_gamma[count_error_col_name] / durations
        )
        binned_gamma = binned_gamma.drop(
            [count_col_name, count_error_col_name], axis=1
        ).copy()
        channel_data[ExperimentDataKey.BINNED_GAMMA] = binned_gamma

In [ ]:
# bin gamma energy spectrum


def make_index_converter(start_time: pd.Timestamp) -> Callable:
    def index_converter(category: pd.Interval) -> pd.Interval:
        cat_start = category.left
        cat_end = category.right
        start_seconds = (cat_start - start_time).total_seconds()
        end_seconds = (cat_end - start_time).total_seconds()
        return pd.Interval(start_seconds, end_seconds, closed=category.closed)

    return index_converter


for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        gamma_report = channel_data[ExperimentDataKey.GAMMA_ONLY]
        time_bins = channel_data[ExperimentDataKey.TIME_BIN_EDGES]
        energy_bins = channel_data[ExperimentDataKey.HISTOGRAM_X_EDGES]

        time_col_name = DetectorDataframeColumn.EVENT_TIME.value
        time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
        eng_bin_col_name = BinningDataframeColumn.ENERGY_BIN.value
        count_col_name = BinningDataframeColumn.COUNT.value

        start_time = time_bins[0]

        binned_gamma_spectrum = get_time_cut(
            gamma_report, time_col_name, time_bins
        )
        energy_cut, energy_bins = pd.cut(
            binned_gamma_spectrum[calibrated_energy_column.value],
            bins=energy_bins,
            retbins=True,
        )
        binned_gamma_spectrum[eng_bin_col_name] = energy_cut
        binned_gamma_spectrum = (
            binned_gamma_spectrum.groupby(
                [time_bin_col_name, eng_bin_col_name],
                as_index=True,
                observed=False,
            )
            .size()
            .to_frame()
            .copy()
        )
        binned_gamma_spectrum.columns = [count_col_name]
        binned_gamma_spectrum = binned_gamma_spectrum.reset_index(level=1)
        binned_gamma_spectrum = binned_gamma_spectrum.pivot_table(
            values=count_col_name,
            index=binned_gamma_spectrum.index,
            columns=eng_bin_col_name,
            observed=False,
        )

        time_index = binned_gamma_spectrum.index
        start_time = time_index[0].left
        convert_categories = make_index_converter(start_time)
        time_index = time_index.map(convert_categories)
        binned_gamma_spectrum.index = time_index

        channel_data[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM] = (
            binned_gamma_spectrum
        )
        channel_data[ExperimentDataKey.GAMMA_ENERGY_BIN_EDGES] = energy_bins

In [ ]:
# process and bin reactor data


def normalize_units(value, unit, to_unit):
    """
    Converts Series of values to desired units

    value: measured value
    units: units of measured value
    to_unit: unit to convert to

    returns Series of values converted to desired unit
    """
    try:
        return Quantity(value, unit).ito(to_unit).magnitude
    except AttributeError:
        return value


for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        time_col_name = NonReactorDataframeColumn.TIME.value
        data_col_name = NonReactorDataframeColumn.DATA.value
        units_col_name = NonReactorDataframeColumn.UNITS.value
        norm_data_col_name = NonReactorDataframeColumn.NORMALIZED_DATA.value
        norm_units_col_name = NonReactorDataframeColumn.NORMALIZED_UNITS.value

        if (
            reactor_data := channel_data.get(ExperimentDataKey.REACTOR_DATA)
        ) is not None:
            binned_reactor_data = {}
            time_bins = channel_data[ExperimentDataKey.TIME_BIN_EDGES]

            for data_source, reactor_param_df in reactor_data.items():
                try:
                    reactor_param_df[data_col_name] = reactor_param_df[
                        data_col_name
                    ].astype(float)
                except ValueError:
                    continue  # skip if not numeric

                units_counts = reactor_param_df[units_col_name].value_counts()
                main_unit = units_counts.idxmax()
                if units_counts.size > 1:
                    # determine most frequent
                    reactor_param_df[norm_data_col_name] = (
                        reactor_param_df.apply(
                            lambda row: normalize_units(
                                row[data_col_name],
                                row[units_col_name],
                                main_unit,
                            ),
                            axis=1,
                        )
                    )
                    reactor_param_df = reactor_param_df.assign(
                        **{norm_units_col_name: lambda _: main_unit}
                    )
                else:
                    reactor_param_df[norm_data_col_name] = reactor_param_df[
                        data_col_name
                    ]
                    reactor_param_df[norm_units_col_name] = reactor_param_df[
                        units_col_name
                    ]

                binned_reactor_param_df = bin_non_neutron_data(
                    reactor_param_df,
                    time_bins,
                    norm_data_col_name,
                    [
                        f"Average {data_source} ({main_unit})",
                        f"{data_source} error ({main_unit})",
                    ],
                )
                binned_reactor_data[data_source] = binned_reactor_param_df
            channel_data[ExperimentDataKey.BINNED_REACTOR_DATA] = (
                binned_reactor_data
            )

In [ ]:
# Merge binned data
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
        bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
        bin_time_col_name = BinningDataframeColumn.BIN_TIME.value

        binned_dfs = []
        binned_dfs.append(channel_data[ExperimentDataKey.BINNED_NEUTRONS])
        binned_dfs.append(channel_data[ExperimentDataKey.BINNED_GAMMA])
        binned_reactor_data = channel_data.get(
            ExperimentDataKey.BINNED_REACTOR_DATA
        )
        if binned_reactor_data is not None:
            binned_reactor_param_dfs = binned_reactor_data.values()
            for binned_reactor_param_df in binned_reactor_param_dfs:
                binned_dfs.append(binned_reactor_param_df)
        merged_df = reduce(
            lambda df1, df2: pd.merge(
                df1,
                df2,
                how="left",
                on=[time_bin_col_name, bin_mid_col_name, bin_time_col_name],
            ),
            binned_dfs,
        )
        channel_data[ExperimentDataKey.ALL_BINNED_DATA] = merged_df

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        neutron_report = channel_data[ExperimentDataKey.NEUTRONS_ONLY]
        max_timetag_hrs = neutron_report[
            DetectorDataframeColumn.TIME_HOURS.value
        ].max()
        print(exp_id, channel, max_timetag_hrs)

## Export and Display

In [ ]:
# clear old data from root, and make new analysis folder
# for exp_name in experiment_neutron_data.keys():
#     root = get_report_root(exp_name)
#     analysis_root = root / analysis_timestamp
#     for file in root.iterdir():
#         if file.is_file() and not file.is_dir():
#             file.unlink()
#     analysis_root.mkdir(parents=True, exist_ok=True)

In [ ]:
# add analysis setting file to experiment root and analysis folder
# for exp_name in experiment_neutron_data.keys():
#     root = get_report_root(exp_name)
#     analysis_root = root / analysis_timestamp
#     file_name = f"{exp_name}_analysis_settings_{analysis_timestamp}.json"
#     with open(root / file_name, 'w') as root_settings_file:
#         json.dump(overall_settings, root_settings_file)
#     # with open(analysis_root / file_name, 'w') as analysis_settings_file:
#     #     json.dump(overall_settings, analysis_settings_file)
#     try:
#         shutil.copy(root / file_name, analysis_root / file_name)
#     except shutil.SameFileError:
#         pass

In [ ]:
# Export as CSV
# for exp_name, data_dict in experiment_neutron_data.items():
#     all_binned_data = data_dict[ExperimentDataKey.ALL_BINNED_DATA]
#     root = get_report_root(exp_name)
#     analysis_root = root / analysis_timestamp
#     file_name = f"{exp_name}_data_{bin_length}s_bin.csv"
#     root_path = root / file_name
#     analysis_path = analysis_root / file_name
#     all_binned_data.to_csv(root_path, index=False)
#     try:
#         shutil.copy(root_path, analysis_path)
#     except shutil.SameFileError:
#         pass
#     print(f"Experiment {exp_name} saved to:")
#     print(f"    - {root_path}")
#     print(f"    - {analysis_path}")

In [ ]:
# for exp_name, data_dict in experiment_neutron_data.items():
#     binned_gamma_spectrum = data_dict[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM]
#     root = get_report_root(exp_name)
#     analysis_root = root / analysis_timestamp
#     file_name = (f"{exp_name}_gamma_spectrum_{bin_length}s_time_bin.csv")
#     root_path = root / file_name
#     analysis_path = analysis_root / file_name
#     binned_gamma_spectrum.to_csv(root_path)
#     try:
#         shutil.copy(root_path, analysis_path)
#     except shutil.SameFileError:
#         pass
#     print(f"Gamma spectrum for {exp_name} saved to:")
#     print(f"    - {root_path}")
#     print(f"    - {analysis_path}")

In [ ]:
# for exp_name, data_dict in experiment_neutron_data.items():
#     psd_report = data_dict[ExperimentDataKey.PSD_REPORT]

#     root = get_report_root(exp_name)
#     analysis_root = root / analysis_timestamp
#     file_name = f"{exp_name}_event_psd_energy.csv"
#     root_path = root / file_name
#     analysis_path = analysis_root / file_name

#     columns = [
#         calibrated_energy_column.value,
#         DetectorDataframeColumn.PSD.value,
#         DetectorDataframeColumn.NEW_N_CLASS.value
#     ]
#     headers = ['Energy (MeVee)', 'PSD', 'Is Neutron?']
#     psd_report.to_csv(
#         root_path,
#         index=False,
#         columns=columns,
#         header=headers
#     )
#     try:
#         shutil.copy(root_path, analysis_path)
#     except shutil.SameFileError:
#         pass
#     print(f"Event energy/PSD data for {exp_name} saved to:")
#     print(f"    - {root_path}")
#     print(f"    - {analysis_path}")

## Diagnostics Display

### Colormaps

In [ ]:
cmap = mpl.colormaps["seismic"]
plot_cmap(cmap)

In [ ]:
seismic_greyred_colors = [
    (0.5, 0.5, 0.5),
    (0.8, 0.8, 0.8),
    (1.0, 1.0, 1.0),
    (1.0, 0.0, 0.0),
    (0.5, 0.0, 0.0),
]
seismic_greyred_cmap = LinearSegmentedColormap.from_list(
    "seismic_greyred", seismic_greyred_colors
)
seismic_greyred_cmap

In [ ]:
seismic_grey_colors = [
    (1.0, 1.0, 1.0),
    (0.8, 0.8, 0.8),
    (0.5, 0.5, 0.5),
]
seismic_grey_cmap = LinearSegmentedColormap.from_list(
    "seismic_grey", seismic_grey_colors
)
seismic_grey_cmap

In [ ]:
plot_cmap(seismic_grey_cmap)

In [ ]:
seismic_red_colors = [
    (1.0, 1.0, 1.0),
    # (0.8, 0.0, 0.0),
    (0.6, 0.0, 0.0),
]
seismic_red_cmap = LinearSegmentedColormap.from_list(
    "seismic_red", seismic_red_colors
)
seismic_red_cmap

In [ ]:
plot_cmap(seismic_red_cmap)

In [ ]:
seismic_blue_colors = [
    (1.0, 1.0, 1.0),
    # (0.0, 0.0, 0.8),
    (0.0, 0.0, 0.6),
]
seismic_blue_cmap = LinearSegmentedColormap.from_list(
    "seismic_blue", seismic_blue_colors
)
seismic_blue_cmap

In [ ]:
plot_cmap(seismic_blue_cmap)

### Plots

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        psd_report = channel_data[ExperimentDataKey.PSD_REPORT]
        fig, ax = plot_scatter(
            psd_report[calibrated_energy_column.value],
            psd_report[DetectorDataframeColumn.PSD.value],
        )
        ax.set_xlabel("Energy [MeVee]", fontsize=14)  # Update x-axis label
        ax.set_ylabel("PSD", fontsize=14)

        ax.set_title(
            f"{exp_id} PSD/Energy Graph (Ch{channel})", ha="center", fontsize=20
        )
        # output_path = get_report_root(exp_name) / f"{exp_name} PSD graph.png"
        from pathlib import Path

        output_path = Path() / f"{exp_id}_{channel} PSD graph.png"
        print(str(output_path))
        fig.savefig(str(output_path))
        plt.show()

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        psd_report = channel_data[ExperimentDataKey.PSD_REPORT]
        fom_results = channel_data[ExperimentDataKey.FOM_RESULTS]
        x_bin_edges = helpers.get_midpoints_from_min_max_series(
            fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MINIMUM.value],
            fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MAXIMUM.value],
            fom_results.index,
        )
        gamma_mu = fom_results.mu1
        gamma_sigma = fom_results.sigma1
        neutron_sigma = fom_results.sigma2

        window_sigma = 5 * gamma_sigma
        fom_sigma = 3 * (gamma_sigma + neutron_sigma)

        fig, ax = plot_scatter(
            psd_report[calibrated_energy_column.value],
            psd_report[DetectorDataframeColumn.PSD.value],
        )
        dot_size = 8
        ax.scatter(
            x_bin_edges,
            window_sigma,
            marker=".",
            linewidths=0,
            s=dot_size,
            label="5 x gamma sigma",
        )
        ax.scatter(
            x_bin_edges,
            fom_sigma,
            marker="o",
            linewidths=0,
            s=dot_size,
            label="3 x sigma sum",
        )
        ax.set_xlabel("Energy [MeVee]", fontsize=14)  # Update x-axis label
        ax.set_ylabel("PSD", fontsize=14)
        # Add a title
        ax.set_title(
            f"{exp_id} PSD/Energy Graph (Ch{channel})", ha="center", fontsize=20
        )
        # output_path = get_report_root(exp_name) / f"{exp_name} PSD graph.png"
        # fig.savefig(output_path)
        plt.legend()
        plt.show()

In [ ]:
### For named color options (and examples), check https://matplotlib.org/stable/gallery/color/named_colors.html#css-colors
# (For more options, check https://matplotlib.org/stable/gallery/color/named_colors.html#css-colors)
# (Any text option (in quotes) can be used)
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        binned_neutrons = channel_data[ExperimentDataKey.ALL_BINNED_DATA]

        bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
        n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
        n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

        zeroed_bins = binned_neutrons[bin_time_col_name] / 60
        rates = binned_neutrons[n_rate_col_name]
        rate_errors = binned_neutrons[n_error_col_name]

        color_block_settings = []
        add_color_blocks = helpers.get_input_with_default(
            "Do you want to add color blocks? [y/N] >", "n", str
        )
        if add_color_blocks.lower() == "y":
            color_blocks_count = helpers.get_input_required(
                "How many color blocks do you want to add? (minimum 1) >",
                (1, None),
                int,
            )
            for i in range(color_blocks_count):
                print(f"For block {i+1}:")
                block_start = helpers.get_input_required(
                    "Where should the block start (in minutes elapsed)? >",
                    (0, None),
                    int,
                )
                block_end = helpers.get_input_required(
                    "Where should the block end (in minutes elapsed)? >",
                    (block_start, None),
                    int,
                )
                block_color = input("What color should the block be? >")
                block_settings = {
                    "start": block_start,
                    "end": block_end,
                    "color": block_color,
                }

                block_needs_text = helpers.get_input_with_default(
                    "Does the block need bottom text? [y/N] >", "n", str
                )
                if block_needs_text.lower() == "y":
                    block_text = input("Enter block text >")
                    block_text_alignment = helpers.get_input_with_default(
                        (
                            "Choose alignment (left, center, right)"
                            "or press Enter for default (center) >"
                        ),
                        "center",
                        str,
                    )
                    block_y_pos = helpers.get_input_required(
                        "Choose text's vertical position (in CPS) >",
                        (0, 160),
                        int,
                    )
                    if block_text_alignment == "left":
                        block_text_position = (block_start, block_y_pos)
                    elif block_text_alignment == "right":
                        block_text_position = (block_end, block_y_pos)
                    else:
                        block_text_alignment = "center"
                        block_text_position = (
                            (block_start + block_end) / 2,
                            block_y_pos,
                        )
                    block_settings["text"] = block_text
                    block_settings["text_alignment"] = block_text_alignment
                    block_settings["text_position"] = block_text_position

                color_block_settings.append(block_settings)

        fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
        dot_size = 8

        ax.errorbar(
            zeroed_bins,
            rates,
            yerr=rate_errors,
            fmt=".",
            linestyle="",
            markersize=dot_size,
            capsize=dot_size,
            color="black",
        )
        ax.set_xlabel("Time [minutes]", fontsize=14)  # Update x-axis label
        ax.set_ylabel("Neutron rate [1/s]", fontsize=14)
        ax.tick_params(labelsize=12)
        ax.set_ylim(0.015, 0.065)
        # ax.set_xlim(0, 145)

        for block_settings in color_block_settings:
            ax.axvspan(
                block_settings["start"],
                block_settings["end"],
                color=block_settings["color"],
                alpha=0.2,
            )
            block_text = block_settings.get("text")
            if block_text is not None:
                text_align = block_settings["text_alignment"]
                text_pos = block_settings["text_position"]
                ax.annotate(
                    block_text,
                    text_pos,
                    horizontalalignment=text_align,
                    fontsize=14,
                )

        --Plot title--
        fig.text(
            0.5,
            0.90,
            f"{exp_name} neutron count rate (Dwell time {bin_length}s)",
            ha='center',
            fontsize=20
        )

        output_path = (
            # get_report_root(exp_name)
            Path()
            / f"{exp_id}-{channel} Count rates {bin_length}s dwell.png"
        )
        fig.savefig(output_path)

        print(f"{exp_id}_{channel}")
        plt.show()

In [ ]:
# import matplotlib as mpl
HISTOGRAM_RES = 1024
COUNT_LIMIT = 20

for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        psd_report = channel_data[ExperimentDataKey.PSD_REPORT]
        borders = channel_data[ExperimentDataKey.BORDERS]

        fig, ax = plot_classification(
            psd_report,
            borders,
            f"{exp_id}-{channel}",
            DetectorDataframeColumn.NEW_N_CLASS,
            calibrated_energy_column,
            count_limit=COUNT_LIMIT,
            # cmap="seismic"
            cmap=seismic_greyred_cmap,
            axis_font_size=28,
            axis_tick_font_size=24,
            line_style="-",
            legend=False,
            # titles=False,
        )

        output_path = (
            # get_report_root(exp_name)
            Path()
            / f"{exp_id}-{channel} Neutron Classification.svg"
        )
        fig.savefig(output_path)

        plt.show()

In [ ]:
low_e_settings = NasaGenerationSettings(
    window_offset=settings.window_offset,
    sigma=settings.sigma,
    lower_energy_bound=0.315,  # equivalent to prior 0.050 MeVee limit
    upper_energy_bound=1.4,
    recalculate_lower_energy_bound=settings.recalculate_lower_energy_bound,
    use_filter=settings.use_filter,
    filter_window=settings.filter_window,
    filter_order=settings.filter_order,
)
low_e_factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, low_e_settings
)
low_e_strategy = low_e_factory_fn()

high_e_settings = NasaGenerationSettings(
    window_offset=settings.window_offset,
    sigma=settings.sigma,
    lower_energy_bound=1.5,
    recalculate_lower_energy_bound=settings.recalculate_lower_energy_bound,
    use_filter=settings.use_filter,
    filter_window=settings.filter_window,
    filter_order=settings.filter_order,
)
high_e_factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, high_e_settings
)
high_e_strategy = high_e_factory_fn()

In [ ]:
count_limit = 5
figsize_x = 10
figsize_y = 8
bin_res_x = 256
bin_res_y = bin_res_x * figsize_x // figsize_y
norm = mpl.colors.Normalize(vmin=0, vmax=count_limit)

for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        psd_report = channel_data[ExperimentDataKey.PSD_REPORT]
        fom_results = channel_data[ExperimentDataKey.FOM_RESULTS]

        low_e_strategy.set_slice_fit_dataframe(fom_results)
        low_e_borders = low_e_strategy.get_neutron_window()
        high_e_strategy.set_slice_fit_dataframe(fom_results)
        high_e_borders = high_e_strategy.get_neutron_window()

        low_e_config = {"cmap": seismic_blue_cmap, "line_color_code": "b"}
        high_e_config = {"cmap": seismic_red_cmap, "line_color_code": "r"}

        fig, ax, counts = plot_multiple_classification(
            psd_report,
            [
                (low_e_borders, "Low E neutrons (2.45 MeV)", low_e_config),
                (high_e_borders, "High E neutrons (14 MeV)", high_e_config),
            ],
            calibrated_energy_column,
            cmap=seismic_grey_cmap,
            legend=False,
        )

        low_e_count, high_e_count, *_ = counts
        print(f"Experiment {exp_id}.{channel}")
        print(f"  - Low E count = {low_e_count}")
        print(f"  - High E count = {high_e_count}")

        suptitle = f"Experiment {exp_id}-{channel}"
        title = f"2.45 MeV count = {low_e_count}, 14 MeV count = {high_e_count}"
        add_supertitle_to_figure(fig, suptitle)
        ax.set_title(title, fontsize=22)

In [ ]:
fom_to_sigma = 2 * sqrt(2 * log(2))

for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        fom_results = channel_data[ExperimentDataKey.FOM_RESULTS]
        slice_mid_energy = helpers.get_midpoints_from_min_max_series(
            fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MINIMUM.value],
            fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MAXIMUM.value],
            fom_results.index,
        ).to_numpy(copy=True)

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.plot(
            slice_mid_energy, fom_results[SliceFitDataframeColumn.FOM.value]
        )
        ax.set_xlabel("Energy [MeVee]", fontsize=14)  # Update x-axis label
        ax.set_ylabel("FOM", fontsize=14)
        # Add a title
        ax.set_title(f"{exp_id}-{channel} FOM Graph", ha="center", fontsize=20)
        ax.hlines(1.27, slice_mid_energy[0], slice_mid_energy[-1], "r", ls="--")

        output_path = (
            # get_report_root(exp_name)
            Path()
            / f"{exp_id}-{channel} Energy vs FOM.png"
        )
        fig.savefig(output_path)
        plt.show()

In [ ]:
# histogram contour plot (vaporwave island)
cmap = plt.colormaps["nipy_spectral"]
figsize = (12, 12)
fontsize = 16
histo_res = 128
contour_res = 250
angle_elev = 30
angle_rot = -60

for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        Z = channel_data[ExperimentDataKey.PSD_HISTOGRAM]
        xe = channel_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
        ye = channel_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
        fig = plt.figure(figsize=figsize)
        ax = plt.axes(projection="3d")
        x, y = np.meshgrid(xe[:-1], ye[:-1])

        ax.view_init(angle_elev, angle_rot)
        ax.contour3D(x, y, Z.T, contour_res, cmap=cmap)
        ax.set_title(
            f"{exp_id}-{channel} PSD/Energy 3D Histogram", fontsize=fontsize + 4
        )
        ax.set_ylabel("PSD", fontsize=fontsize)
        ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
        ax.set_zlabel("Counts", fontsize=fontsize)

        output_path = (
            # get_report_root(exp_name)
            Path()
            / f"{exp_id}-{channel} PSD 3D Histogram.png"
        )
        fig.savefig(output_path)

        plt.show()

In [ ]:
cmap = plt.colormaps["nipy_spectral"]
figsize = (12, 12)
fontsize = 16
contour_res = 50
angle_elev = 30
angle_rot = -60
ls = LightSource(270, 45)

for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        Z = channel_data[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM].to_numpy()
        xe = channel_data[ExperimentDataKey.GAMMA_ENERGY_BIN_EDGES]
        # TODO find x edge closest to high cutoff (0.5)
        time_bins = channel_data[ExperimentDataKey.TIME_BIN_EDGES]
        start_time = time_bins[0]
        ye = (time_bins - start_time).total_seconds() / 3600
        xmid = (xe[1:] + xe[:-1]) / 2
        ymid = (ye[1:] + ye[:-1]) / 2
        # xmid = xe[:-1]
        # ymid = ye[:-1]
        if xmid.shape[0] == 1:
            print(
                f"{exp_id}_{channel}:",
                "Energy axis has too few points to make a surface",
            )
            continue
        if ymid.shape[0] == 1:
            print(
                f"{exp_id}_{channel}:",
                "Time axis has too few points to make a surface",
            )
            continue

        fig = plt.figure(figsize=figsize)
        ax = plt.axes(projection="3d")
        x, y = np.meshgrid(xmid, ymid)
        print(x.shape)
        print(y.shape)

        ax.view_init(angle_elev, angle_rot)
        rgb = ls.shade(Z, cmap=cmap, blend_mode="soft")
        # ax.contour3D(x, y, Z, contour_res, cmap=cmap)
        ax.plot_surface(
            x,
            y,
            Z,
            rstride=1,
            cstride=1,
            facecolors=rgb,
            linewidth=0,
            antialiased=False,
            shade=False,
        )
        ax.set_title(
            f"{exp_id}-{channel} PSD/Energy 3D Histogram", fontsize=fontsize + 4
        )
        ax.set_ylabel("Time (hours)", fontsize=fontsize)
        ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
        ax.set_zlabel("Counts", fontsize=fontsize)

        output_path = (
            # get_report_root(exp_name)
            Path()
            / f"{exp_id}-{channel} Gamma Spectrum 3D Histogram.png"
        )
        fig.savefig(output_path)

        plt.show()

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    channels_data = exp_data[ExperimentDataKey.BY_CHANNEL]
    for channel, channel_data in channels_data.items():
        gamma_energy_spectrum_df = channel_data[
            ExperimentDataKey.GAMMA_ENERGY_SPECTRUM
        ]
        do_spectrum = input(
            f"Do you want a gamma spectrum for {exp_id}-{channel}? [y/N] "
        )
        if do_spectrum.lower() not in ["y", "yes"]:
            continue
        elapsed_str = input(
            "At what time (elapsed hours) do you want to get the spectrum? "
        )
        try:
            spectrum_time_elapsed = float(elapsed_str)
        except ValueError:
            print(f"The value {elapsed_str} was not a valid decimal number")
            continue
        time_bins = channel_data[ExperimentDataKey.TIME_BIN_EDGES]
        start_time = time_bins[0]
        spectrum_timestamp = start_time + timedelta(hours=spectrum_time_elapsed)

        matching_intervals = [
            interval
            for interval in gamma_energy_spectrum_df.index.categories
            if spectrum_timestamp in interval
        ]
        if len(matching_intervals) == 0:
            print(
                (
                    f"The given time ({spectrum_time_elapsed} hrs) "
                    + "has no match in this experiment"
                )
            )
            continue
        matching_interval = matching_intervals[0]
        gamma_test_spectrum = gamma_energy_spectrum_df[
            gamma_energy_spectrum_df.index == matching_interval
        ]

        gamma_energy_bins = gamma_test_spectrum.T.index.to_series()
        midpoints = gamma_energy_bins.apply(lambda x: x.mid)
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.scatter(midpoints, gamma_test_spectrum.T, s=1)
        ax.set_title(
            f"Gamma Energy Spectrum - {exp_id}-{channel} "
            f"@ {spectrum_time_elapsed} hrs"
        )
        ax.set_xlabel("Energy (MeVee)")
        ax.set_ylabel("Count")

        plt.show()

## Done!

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()

### Post-Completion

In [ ]:
exp_id = "TB-97"
data_dict = experiment_neutron_data[exp_id]
[x for x in detector_setup.keys()]
# neutron_report = data_dict[ExperimentDataKey.NEUTRONS_ONLY]

# print(neutron_report.head())

In [ ]:
# For TB-97, get rate for only high energy neutrons
exp_id = "TB-97"
data_dict = experiment_neutron_data[exp_id]
channels_data = data_dict[ExperimentDataKey.BY_CHANNEL]
for channel in detector_setup.keys():
    channel_data = channels_data[channel]
    neutron_report = channel_data[ExperimentDataKey.NEUTRONS_ONLY]
    time_bins = channel_data[ExperimentDataKey.TIME_BIN_EDGES]
    
    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    energy_col_name = calibrated_energy_column.value
    psd_col_name = DetectorDataframeColumn.PSD.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value
    count_error_col_name = BinningDataframeColumn.COUNT_ERROR.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value
    
    # select only neutrons with L > 2.5
    # high_e_neutrons = neutron_report[neutron_report[calibrated_energy_column.value] > 2.5]
    high_e_neutrons = neutron_report.query(
        f"`{energy_col_name}` >= 2.45 & `{psd_col_name}` > 0.2 & `{psd_col_name}` <= 0.3"
    )
    # use time cut (already done!) to bin neutrons by time bin
    
    binned_neutrons = (
        high_e_neutrons.groupby(time_bin_col_name, as_index=True)
        .size()
        .to_frame()
        .copy()
    )
    binned_neutrons.columns = [count_col_name]
    binned_neutrons[count_error_col_name] = np.sqrt(binned_neutrons[count_col_name])
    
    binned_neutron_time_bins = binned_neutrons.index.to_series()
    midpoints = binned_neutron_time_bins.apply(lambda x: x.mid)
    durations = binned_neutron_time_bins.apply(
        lambda x: x.length.total_seconds()
    ).astype(np.float64)
    
    binned_neutrons[bin_mid_col_name] = midpoints
    binned_neutrons = bin_midpoint_time_to_seconds(binned_neutrons, start_time)
    
    binned_neutrons[n_rate_col_name] = binned_neutrons[count_col_name] / durations
    binned_neutrons[n_error_col_name] = (
        # binned_neutrons[count_error_col_name] / durations
        binned_neutrons[n_rate_col_name]
        / np.sqrt(binned_neutrons[count_col_name])
    )
    # binned_neutrons = binned_neutrons.drop(
    #     [count_col_name, count_error_col_name],
    #     axis=1
    # ) \
    #     .copy()
    channel_data[ExperimentDataKey.BINNED_NEUTRONS] = binned_neutrons

In [ ]:
bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

for channel, channel_data in channels_data.items():
    binned_neutrons = channel_data[ExperimentDataKey.BINNED_NEUTRONS]
    zeroed_bins = binned_neutrons[bin_time_col_name] / 60
    rates = binned_neutrons[n_rate_col_name]
    rate_errors = binned_neutrons[n_error_col_name]
    
    color_block_settings = []
    add_color_blocks = helpers.get_input_with_default(
        "Do you want to add color blocks? [y/N] >", "n", str
    )
    if add_color_blocks.lower() == "y":
        color_blocks_count = helpers.get_input_required(
            "How many color blocks do you want to add? (minimum 1) >",
            (1, None),
            int,
        )
        for i in range(color_blocks_count):
            print(f"For block {i+1}:")
            block_start = helpers.get_input_required(
                "Where should the block start (in minutes elapsed)? >",
                (0, None),
                float,
            )
            block_end = helpers.get_input_required(
                "Where should the block end (in minutes elapsed)? >",
                (block_start, None),
                float,
            )
            block_color = input("What color should the block be? >")
            block_settings = {
                "start": block_start,
                "end": block_end,
                "color": block_color,
            }
    
            block_needs_text = helpers.get_input_with_default(
                "Does the block need bottom text? [y/N] >", "n", str
            )
            if block_needs_text.lower() == "y":
                block_text = input("Enter block text >")
                block_text_alignment = helpers.get_input_with_default(
                    "Choose alignment (left, center, right) or press Enter for default (center) >",
                    "center",
                    str,
                )
                block_y_pos = helpers.get_input_required(
                    "Choose text's vertical position (in CPS) >", (0, 160), int
                )
                if block_text_alignment == "left":
                    block_text_position = (block_start, block_y_pos)
                elif block_text_alignment == "right":
                    block_text_position = (block_end, block_y_pos)
                else:
                    block_text_alignment = "center"
                    block_text_position = (
                        (block_start + block_end) / 2,
                        block_y_pos,
                    )
                block_settings["text"] = block_text
                block_settings["text_alignment"] = block_text_alignment
                block_settings["text_position"] = block_text_position
    
            color_block_settings.append(block_settings)
    
    fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
    dot_size = 8
    
    ax.errorbar(
        zeroed_bins,
        rates,
        yerr=rate_errors,
        fmt=".",
        linestyle="",
        markersize=dot_size,
        capsize=dot_size,
        color="black",
    )
    ax.set_xlabel("Time [minutes]", fontsize=14)  # Update x-axis label
    ax.set_ylabel("Neutron rate [1/s]", fontsize=14)
    ax.tick_params(labelsize=12)
    # ax.set_ylim(0, 150)
    # ax.set_xlim(0, 145)
    
    for block_settings in color_block_settings:
        ax.axvspan(
            block_settings["start"],
            block_settings["end"],
            color=block_settings["color"],
            alpha=0.2,
        )
        block_text = block_settings.get("text")
        if block_text is not None:
            text_align = block_settings["text_alignment"]
            text_pos = block_settings["text_position"]
            ax.annotate(
                block_text, text_pos, horizontalalignment=text_align, fontsize=14
            )
    
    fig.text(
        0.5,
        0.90,
        f"{exp_id}-{channel} HE count rate (Dwell {bin_length}s)",
        ha='center',
        fontsize=20
    )
    
    # Save the plot
    # output_path = (
    #     # get_report_root(exp_name)
    #     Path()
    #     / f"{exp_name} Count rates {bin_length}s dwell.png"
    # )
    # fig.savefig(output_path)
    
    # Show the plot (optional)
    plt.show()

In [ ]:
# ta cleaning I want you to leave only points that fit within these phases, and only then plot time series:
# Phase 0 – Background: start 17.8 min, end 77.8 min.
# Phase 1 – Beam Loading: start 158.8 min, end 218.8 min.
# Phase 2 – BL + Ecell: start 218.8 min, end 278.8 min.
# Phase 3 – Preload: start 278.8 min, end 338.8 min.

In [ ]:
phases = [(17.8, 77.8), (158.8, 218.8), (218.8, 278.8), (278.8, 338.8)]
start_time = data_dict[ExperimentDataKey.START_TIME]
phases_seconds = [tuple([t * 60 for t in phase]) for phase in phases]
phases_datetime = [
    tuple([start_time + timedelta(seconds=t) for t in phase])
    for phase in phases_seconds
]
phases_timestamp = [
    tuple([pd.Timestamp(t) for t in phase]) for phase in phases_datetime
]
phases_intervals = [pd.Interval(*phase) for phase in phases_timestamp]

phase_cut = pd.cut(high_e_neutrons[time_col_name], bins=phases_intervals)
high_e_neutrons.loc[:, "PHASE_BIN"] = phase_cut

binned_high_e_neutrons = (
    high_e_neutrons.groupby("PHASE_BIN", as_index=True).size().to_frame().copy()
)
binned_high_e_neutrons.columns = [count_col_name]

binned_high_e_neutrons[count_error_col_name] = np.sqrt(
    binned_high_e_neutrons[count_col_name]
)

binned_neutron_time_bins = binned_high_e_neutrons.index.to_series()
midpoints = binned_neutron_time_bins.apply(lambda x: x.mid)
durations = binned_neutron_time_bins.apply(
    lambda x: x.length.total_seconds()
).astype(np.float64)

binned_high_e_neutrons[bin_mid_col_name] = midpoints
binned_high_e_neutrons = bin_midpoint_time_to_seconds(
    binned_high_e_neutrons, start_time
)

binned_high_e_neutrons[n_rate_col_name] = (
    binned_high_e_neutrons[count_col_name] / durations
)
binned_high_e_neutrons[n_error_col_name] = (
    # binned_neutrons[count_error_col_name] / durations
    binned_high_e_neutrons[n_rate_col_name]
    / np.sqrt(binned_high_e_neutrons[count_col_name])
)

binned_high_e_neutrons.index = [
    "Background",
    "Phase I",
    "Phase II",
    "Phase III",
]

In [ ]:
bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

bins = binned_high_e_neutrons.index
rates = binned_high_e_neutrons[n_rate_col_name]
rate_errors = binned_high_e_neutrons[n_error_col_name]

x = np.arange(len(bins))

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
dot_size = 8

ax.errorbar(
    x,
    rates,
    yerr=rate_errors,
    fmt=".",
    linestyle="",
    markersize=dot_size,
    capsize=dot_size,
    color="black",
)
ax.set_xticks(x)
ax.set_xticklabels(bins)
ax.set_xlim(-0.5, len(bins) - 0.5)
# ax.set_xlabel("Time [minutes]", fontsize=14)
ax.set_ylabel("Neutron rate [1/s]", fontsize=14)
ax.tick_params(labelsize=12)

ax.axvspan(-0.5, 0.5, color="grey", alpha=0.2, zorder=0)
ax.axvspan(1.5, 2.5, color="grey", alpha=0.2, zorder=0)
ax.grid(axis="y", color="black", alpha=0.3, lw=0.25)

In [ ]:
# phase_bin_dfs = []
# for phase in phases:
#     corrected_times = [time * 60 * 1E12 for time in phase]
#     start, end = corrected_times
#     phase_df = high_e_neutrons.query("TIMETAG > @start & TIMETAG <= @end")

#     binned_neutrons = phase_df.groupby(
#         time_bin_col_name, as_index=True) \
#         .size() \
#         .to_frame() \
#         .copy()
#     binned_neutrons.columns = [count_col_name]
#     binned_neutrons[count_error_col_name] = np.sqrt(
#         binned_neutrons[count_col_name]
#     )

#     binned_neutron_time_bins = binned_neutrons.index.to_series()
#     midpoints = binned_neutron_time_bins.apply(lambda x: x.mid)
#     durations = binned_neutron_time_bins.apply(
#         lambda x: x.length.total_seconds()
#     ).astype(np.float64)

#     binned_neutrons[bin_mid_col_name] = midpoints
#     binned_neutrons = bin_midpoint_time_to_seconds(binned_neutrons, start_time)

#     binned_neutrons[n_rate_col_name] = (
#         binned_neutrons[count_col_name] / durations)
#     binned_neutrons[n_error_col_name] = (
#         # binned_neutrons[count_error_col_name] / durations
#         binned_neutrons[n_rate_col_name] / np.sqrt(binned_neutrons[count_col_name])
#     )
#     phase_bin_dfs.append(binned_neutrons)

In [ ]:
# bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
# n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
# n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

# for phase_df in phase_bin_dfs:
#     zeroed_bins = phase_df[bin_time_col_name] / 60
#     rates = phase_df[n_rate_col_name]
#     rate_errors = phase_df[n_error_col_name]

#     fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
#     dot_size = 8

#     ax.errorbar(
#         zeroed_bins,
#         rates,
#         yerr=rate_errors,
#         fmt=".",
#         linestyle='',
#         markersize=dot_size,
#         capsize=dot_size,
#         color="black"
#     )
#     ax.set_xlabel("Time [minutes]", fontsize=14)  # Update x-axis label
#     ax.set_ylabel("Neutron rate [1/s]", fontsize=14)
#     ax.tick_params(labelsize=12)

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()

In [ ]:
list(experiment_neutron_data.keys())

In [ ]:
labels = {
    "ID-479": "June - 2 hours",
    "TB-46": "September - 0.5 hours (with KOH)",
    # "TB-47": "October - 24 hours"
    "TB-47": "24 hours background at UBC",
}

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
dot_size = 8

for exp_id, data_dict in experiment_neutron_data.items():
    # data_dict = experiment_neutron_data["ID-479"]
    binned_neutrons = data_dict[ExperimentDataKey.ALL_BINNED_DATA]

    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

    zeroed_bins = binned_neutrons[bin_time_col_name] / 60
    rates = binned_neutrons[n_rate_col_name]
    rate_errors = binned_neutrons[n_error_col_name]

    hist, edges = np.histogram(rates, bins="fd")
    midpoints = (edges[1:] + edges[:-1]) / 2

    hist_mask = hist == 0
    hist = hist[~hist_mask]
    midpoints = midpoints[~hist_mask]
    data_dict["rate_histo"] = hist
    data_dict["rate_bin_mids"] = midpoints
    data_dict["rate_bin_edges"] = edges

    ax.scatter(midpoints, hist, label=labels.get(exp_id, "???"))
    ax.set(xlabel="Neutron rate (counts per second)", ylabel="N")
ax.legend()
plt.show()

In [ ]:
from data_processing.processing.figure_of_merit import gaussian
from scipy.optimize import curve_fit

for exp_id, data_dict in experiment_neutron_data.items():
    edges = data_dict["rate_bin_edges"]
    midpoints = data_dict["rate_bin_mids"]
    hist = data_dict["rate_histo"]

    guess_A = max(hist)
    max_idx = np.where(hist == guess_A)[0][0]
    guess_mu = midpoints[max_idx]
    guess_sigma = (edges[-1] - edges[0]) / 4

    params, cov = curve_fit(
        gaussian, midpoints, hist, p0=(guess_mu, guess_sigma, guess_A)
    )
    print(exp_id)
    print(params)

    data_dict["hist_fit_params"] = params
    data_dict["hist_fit_cov"] = cov

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
dot_size = 8

for exp_id, data_dict in experiment_neutron_data.items():
    edges = data_dict["rate_bin_edges"]
    params = data_dict["hist_fit_params"]
    midpoints = data_dict["rate_bin_mids"]
    hist = data_dict["rate_histo"]

    mu, sigma, A = params

    x = np.linspace(edges[0], edges[-1], num=100)
    y = gaussian(x, *params)

    ax.scatter(midpoints, hist, label=labels.get(exp_id, "???"), color="black")
    ax.plot(x, y, color="black")
    # ax.vlines(mu, 0, A, color="black", linestyle="dashed")
    # ax.hlines(A / 2, mu-sigma, mu+sigma, color="black", linestyle="dashed")
    # ax.text(mu, A, f"Mean: {mu:.2f} cps", horizontalalignment="center", bbox={"boxstyle": "round", "color": "#FFFFFFD0"})
    # ax.text(mu, A/2+5, f"SD: {sigma:.2f} cps", horizontalalignment="center", bbox={"boxstyle": "round", "color": "#FFFFFFD0"})

ax.set(xlabel="Neutron production rate (1/s)", ylabel="Counts")
ax.legend()
plt.show()

In [ ]:
experiment_neutron_data["ID-479"][ExperimentDataKey.PSD_REPORT][
    "TIMETAG"
].max() / 10e12 / 60 / 60

In [ ]:
experiment_neutron_data["TB-46"][ExperimentDataKey.PSD_REPORT][
    "TIMETAG"
].max() / 10e12 / 60 / 60